In [1]:
import pandas as pd
import numpy as np
import networkx as nx
from scipy import stats


In [18]:

# -----------------------------------------------------------------------------
# 1. FILE PATHS & SETUP
# -----------------------------------------------------------------------------
EDGE_TABLE_PATH = "data/CMB-2216/edge_table_trend.csv"
NODE_TABLE_PATH = "data/CMB-2216/node_table_trend.csv"
N_SIMULATIONS = 1000

df_edges = pd.read_csv(EDGE_TABLE_PATH)
df_nodes = pd.read_csv(NODE_TABLE_PATH)

G_real = nx.from_pandas_edgelist(
    df_edges,
    source="node1_string_id",
    target="node2_string_id"
)

nodes = list(G_real.nodes())
num_edges = len(G_real.edges())

# -----------------------------------------------------------------------------
# 2. HELPER FUNCTIONS
# -----------------------------------------------------------------------------
def calculate_all_centralities(G):
    """Betweenness, Closeness, and Clustering Coefficient."""
    betweenness = nx.betweenness_centrality(G)
    closeness = nx.closeness_centrality(G)
    clustering = nx.clustering(G)
    return betweenness, closeness, clustering


def gini_coefficient(values):
    """Gini coefficient of a non-negative array (0 = perfectly even, 1 = maximally concentrated)."""
    arr = np.sort(np.asarray(values, dtype=float))
    n = len(arr)
    if n == 0 or arr.sum() == 0:
        return 0.0
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * arr)) / (n * arr.sum()) - (n + 1) / n


def holm_bonferroni(pvals, alpha=0.05):
    """Holm step-down correction. Returns (adjusted_pvals, reject) aligned to input order."""
    pvals = np.asarray(pvals)
    n = len(pvals)
    order = np.argsort(pvals)
    adjusted = np.empty(n)
    running_max = 0.0
    for rank, idx in enumerate(order):
        adj = min((n - rank) * pvals[idx], 1.0)
        running_max = max(running_max, adj)
        adjusted[idx] = running_max
    return adjusted, adjusted < alpha

# -----------------------------------------------------------------------------
# 3. RUN DEGREE-PRESERVING RANDOMIZATIONS
# -----------------------------------------------------------------------------
null_betweenness = {node: [] for node in nodes}   # per-node null values (Section 4)
null_closeness = {node: [] for node in nodes}
null_clustering = {node: [] for node in nodes}

# One independent value PER SIMULATED NETWORK (not per node) for each metric's
# dispersion -- consumed by Section 5's global test instead of pooling
# num_nodes x N_SIMULATIONS node-level values together.
null_var = {'betweenness': [], 'closeness': [], 'clustering': []}
null_gini = {'betweenness': [], 'closeness': [], 'clustering': []}

swap_failures = 0

print(f"Running {N_SIMULATIONS} degree-preserving randomizations in memory...")

for i in range(N_SIMULATIONS):
    G_rand = G_real.copy()

    try:
        nx.double_edge_swap(G_rand, nswap=num_edges * 10, max_tries=num_edges * 100)
    except (nx.NetworkXError, nx.NetworkXAlgorithmError):
        swap_failures += 1

    rand_b, rand_c, rand_clust = calculate_all_centralities(G_rand)

    for node in nodes:
        null_betweenness[node].append(rand_b[node])
        null_closeness[node].append(rand_c[node])
        null_clustering[node].append(rand_clust[node])

    null_var['betweenness'].append(np.var(list(rand_b.values())))
    null_var['closeness'].append(np.var(list(rand_c.values())))
    null_var['clustering'].append(np.var(list(rand_clust.values())))

    null_gini['betweenness'].append(gini_coefficient(list(rand_b.values())))
    null_gini['closeness'].append(gini_coefficient(list(rand_c.values())))
    null_gini['clustering'].append(gini_coefficient(list(rand_clust.values())))

if swap_failures > 0:
    print(f"WARNING: double_edge_swap did not reach the target swap count in "
          f"{swap_failures}/{N_SIMULATIONS} simulations (max_tries exceeded). "
          f"Those networks are only partially randomized. If this count is "
          f"large, increase max_tries or lower nswap.")

print("Randomization complete! Calculating statistical significance...\n")

# -----------------------------------------------------------------------------
# 4. NODE-LEVEL Z-SCORES AND EMPIRICAL P-VALUES
# -----------------------------------------------------------------------------
real_b, real_c, real_clust = calculate_all_centralities(G_real)

results = []
for node in nodes:
    b_obs = real_b[node]
    b_null = np.array(null_betweenness[node])
    b_mean, b_std = np.mean(b_null), np.std(b_null)
    b_z = (b_obs - b_mean) / b_std if b_std > 0 else 0.0
    b_pval = (np.sum(b_null >= b_obs) + 1) / (N_SIMULATIONS + 1)

    c_obs = real_c[node]
    c_null = np.array(null_closeness[node])
    c_mean, c_std = np.mean(c_null), np.std(c_null)
    c_z = (c_obs - c_mean) / c_std if c_std > 0 else 0.0
    c_pval = (np.sum(c_null >= c_obs) + 1) / (N_SIMULATIONS + 1)

    clust_obs = real_clust[node]
    clust_null = np.array(null_clustering[node])
    clust_mean, clust_std = np.mean(clust_null), np.std(clust_null)
    clust_z = (clust_obs - clust_mean) / clust_std if clust_std > 0 else 0.0
    clust_pval = (np.sum(clust_null >= clust_obs) + 1) / (N_SIMULATIONS + 1)

    results.append({
        'STRING_ID': node,
        'Observed_Betweenness': b_obs, 'Betweenness_Null_Mean': b_mean,
        'Betweenness_ZScore': b_z, 'Betweenness_pValue': b_pval,
        'Observed_Closeness': c_obs, 'Closeness_Null_Mean': c_mean,
        'Closeness_ZScore': c_z, 'Closeness_pValue': c_pval,
        'Observed_ClusteringCoeff': clust_obs, 'ClusteringCoeff_Null_Mean': clust_mean,
        'ClusteringCoeff_ZScore': clust_z, 'ClusteringCoeff_pValue': clust_pval
    })

df_stats = pd.DataFrame(results)

# -----------------------------------------------------------------------------
# 5. GLOBAL NETWORK DISPERSION TESTS (Monte Carlo, degree-preserved null)
# -----------------------------------------------------------------------------
# Each test compares ONE summary value (variance, and Gini for skew-
# robustness) computed on the real network against N_SIMULATIONS independent
# summary values, one per randomized network -- matching the number of
# independent random draws actually performed, rather than pooling every
# node's value across every simulation into one (non-independent) sample.

real_var = {
    'betweenness': np.var(list(real_b.values())),
    'closeness': np.var(list(real_c.values())),
    'clustering': np.var(list(real_clust.values())),
}
real_gini_vals = {
    'betweenness': gini_coefficient(list(real_b.values())),
    'closeness': gini_coefficient(list(real_c.values())),
    'clustering': gini_coefficient(list(real_clust.values())),
}

global_results = []
for metric in ['betweenness', 'closeness', 'clustering']:
    for stat_name, real_val, null_vals in [
        ('Variance', real_var[metric], np.array(null_var[metric])),
        ('Gini', real_gini_vals[metric], np.array(null_gini[metric])),
    ]:
        # Upper tail: is the real network MORE concentrated/heterogeneous
        # than the degree-preserved null predicts?
        pval = (np.sum(null_vals >= real_val) + 1) / (N_SIMULATIONS + 1)
        global_results.append({
            'Metric': metric,
            'Statistic': stat_name,
            'Observed': real_val,
            'Null_Mean': null_vals.mean(),
            'pValue_raw': pval,
        })

df_global = pd.DataFrame(global_results)
# Small number of tests (6) -- Holm-Bonferroni is a reasonable correction here.
df_global['pValue_holm'], df_global['Significant_Holm'] = holm_bonferroni(
    df_global['pValue_raw'].values
)

OUTPUT_GLOBAL_FILE = "Results/CMB2216/global_dispersion_tests_trend.csv"
df_global.to_csv(OUTPUT_GLOBAL_FILE, index=False)

# -----------------------------------------------------------------------------
# 6. MERGE NODE-LEVEL RESULTS WITH METADATA & EXPORT
# -----------------------------------------------------------------------------
if 'string_id' in df_nodes.columns:
    df_final = pd.merge(df_nodes, df_stats, left_on='string_id', right_on='STRING_ID', how='inner')
else:
    df_final = df_stats

OUTPUT_FILE = "Results/CMB2216/topological_significance_summary_trend.csv"
df_final.to_csv(OUTPUT_FILE, index=False)

# -----------------------------------------------------------------------------
# 7. PRINT CONSOLE SUMMARY REPORT
# -----------------------------------------------------------------------------
sig_b = df_stats[df_stats['Betweenness_pValue'] < 0.05]
sig_c = df_stats[df_stats['Closeness_pValue'] < 0.05]
sig_clust = df_stats[df_stats['ClusteringCoeff_pValue'] < 0.05]

print("=" * 70)
print("              TOPOLOGICAL ANALYSIS SUMMARY REPORT")
print("=" * 70)
print(f"Total Nodes Analyzed   : {len(nodes)}")
print(f"Random Networks Built  : {N_SIMULATIONS}  (partial swaps: {swap_failures})")
print(f"Node-Level Results File: {OUTPUT_FILE}")
print(f"Global Test Results    : {OUTPUT_GLOBAL_FILE}")
print("-" * 70)
print("GLOBAL DISPERSION TESTS (Monte Carlo, degree-preserved null, Holm-corrected):")
for _, r in df_global.iterrows():
    flag = "*" if r['Significant_Holm'] else " "
    print(f"  {flag} {r['Metric']:<12} {r['Statistic']:<9} | "
          f"obs={r['Observed']:.4f} null_mean={r['Null_Mean']:.4f} | "
          f"p_raw={r['pValue_raw']:.4f}  p_holm={r['pValue_holm']:.4f}")
print("-" * 70)


Running 1000 degree-preserving randomizations in memory...
Randomization complete! Calculating statistical significance...

              TOPOLOGICAL ANALYSIS SUMMARY REPORT
Total Nodes Analyzed   : 15
Random Networks Built  : 1000  (partial swaps: 0)
Node-Level Results File: Results/CMB2216/topological_significance_summary_trend.csv
Global Test Results    : Results/CMB2216/global_dispersion_tests_trend.csv
----------------------------------------------------------------------
GLOBAL DISPERSION TESTS (Monte Carlo, degree-preserved null, Holm-corrected):
    betweenness  Variance  | obs=0.0003 null_mean=0.0103 | p_raw=0.9820  p_holm=1.0000
  * betweenness  Gini      | obs=0.8242 null_mean=0.5971 | p_raw=0.0010  p_holm=0.0060
    closeness    Variance  | obs=0.0009 null_mean=0.0039 | p_raw=1.0000  p_holm=1.0000
    closeness    Gini      | obs=0.0927 null_mean=0.1796 | p_raw=1.0000  p_holm=1.0000
  * clustering   Variance  | obs=0.2400 null_mean=0.0219 | p_raw=0.0010  p_holm=0.0060
    c